# 10 - One analysed picture per landing (Kaggle)

Renders the 5 m decision view for every landing in the finished Monte Carlo (see `touchdown/analysis/render_pictures.py`) and writes `/kaggle/working/pictures.zip` for `web/public/data`.

Settings: **GPU T4 x2**, **Internet on**. Inputs: the private dataset `touchdown-detector-v2` (the detector) and the output of the faithful Monte Carlo notebook (07), added as a notebook output input so `landing_*.json` files are visible under `/kaggle/input`. About 1 s per landing, so roughly 20 to 30 minutes for 1,000.

In [ ]:
!git clone -q https://github.com/aryanyaksh-art/touch_down.git /tmp/touch_down || (cd /tmp/touch_down && git pull -q)
!sh /tmp/touch_down/scripts/kaggle_setup.sh 2>&1 | tail -5

In [ ]:
import glob, os
# the v2 detector was uploaded as two parts; join them back
!find /kaggle/input -name "best.pt.part*" | sort | xargs cat > /tmp/best.pt
CKPT = '/tmp/best.pt'
assert os.path.getsize(CKPT) == 17520863, os.path.getsize(CKPT)
found = sorted(glob.glob('/kaggle/input/**/landing_*.json', recursive=True))
assert found, 'add the faithful Monte Carlo output as an input'
MC = os.path.dirname(found[0])
print(len(found), 'landings in', MC)

In [ ]:
cmd = ('cd /tmp/touch_down && TOUCHDOWN_DATA=/tmp/data TOUCHDOWN_BLENDER=/tmp/blender/blender '
       f'python -m touchdown.analysis.render_pictures --mc {MC} --out /kaggle/working/web_data --device GPU '
       f'--checkpoint {CKPT} --max-hours 8')
!{cmd}
# the original summary.json predates the bootstrap intervals and paired tests, so recompute it from the same landings
!cd /tmp/touch_down && TOUCHDOWN_DATA=/tmp/data python -m touchdown.analysis.compare_published {MC} --out /kaggle/working/web_data/summary.json | tail -3
!cd /kaggle/working && zip -qr pictures.zip web_data && ls -la pictures.zip && du -sh web_data